<a href="https://colab.research.google.com/github/G0nkly/pytorch_sandbox/blob/main/gpts/nanoGPT/AK_GPT_DIY_RECAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Load the data
# Encoding / Decoding
# Train/Test Split | get_batch function

In [2]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-09-20 06:10:01--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  6.30MB/s    in 0.2s    

2026-09-20 06:10:02 (6.30 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [42]:
import torch
import torch.nn.functional as F
import torch.nn as nn

In [43]:
with open("./input.txt") as file:
  text = file.read()

In [44]:
vocab = sorted(set([c for c in text.strip()]))
stoi = {v: k  for k,v in enumerate(vocab)}
itos = {k: v  for k,v in enumerate(vocab)}
encode = lambda chars: [stoi[char] for char in chars]
decode = lambda indices: "".join(itos[index] for index in indices)

In [45]:
###################
# HYPERPARAMETERS #
###################
vocab_size = len(vocab)
block_size = 8
batch_size = 8
embed_dim = 32
n_heads = 4
eval_iter = 1000
train_test_split = 0.8
device = "cuda" if torch.cuda.is_available() else "cpu"

In [46]:
data = torch.tensor(encode(text))

In [47]:
data[:10]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])

In [48]:
train_index = int(train_test_split * len(data))
train = data[:train_index]
test = data[train_index:]
indices = torch.randint(0, len(train) - block_size, (batch_size,))
indices

tensor([732307, 761394, 110855, 886048, 856573,  20361, 788372, 617043])

In [49]:
indices = torch.randint(0, len(train) - block_size,(1,8))
indices = indices.squeeze(0).tolist()
x = []
y = []
for index in indices:
  x.append(torch.stack([train[index + i] for i in range(0, block_size)]))
  y.append(torch.stack([train[index + i + 1] for i in range(0, block_size)]))

X = torch.stack(x)
Y = torch.stack(y)

X, Y


(tensor([[41, 46,  1, 46, 39, 60, 43,  6],
         [52, 45, 43, 56,  1, 58, 46, 59],
         [58,  1, 47, 52,  1, 46, 39, 64],
         [ 1, 57, 53,  1, 50, 47, 58, 58],
         [53, 52,  6,  0, 32, 46, 39, 58],
         [56,  1, 42, 43, 39, 58, 46, 57],
         [53, 59, 56,  1, 54, 56, 43, 57],
         [42,  1, 58, 46, 43, 63,  1, 57]]),
 tensor([[46,  1, 46, 39, 60, 43,  6,  1],
         [45, 43, 56,  1, 58, 46, 59, 57],
         [ 1, 47, 52,  1, 46, 39, 64, 39],
         [57, 53,  1, 50, 47, 58, 58, 50],
         [52,  6,  0, 32, 46, 39, 58,  1],
         [ 1, 42, 43, 39, 58, 46, 57,  8],
         [59, 56,  1, 54, 56, 43, 57, 43],
         [ 1, 58, 46, 43, 63,  1, 57, 46]]))

In [50]:
def get_batch(split = "train"):
  data = train if split == "train" else test
  indices = torch.randint(0, len(train) - block_size, (batch_size,))

  x = []
  y = []
  for index in indices:
    x.append(torch.stack([train[index + i] for i in range(0, block_size)]))
    y.append(torch.stack([train[index + i + 1] for i in range(0, block_size)]))

  X = torch.stack(x)
  Y = torch.stack(y)

  return X, Y

In [51]:
@torch.no_grad()
def evaluate_model(model):
  model.eval()
  losses = torch.zeros(eval_iter)
  for i in range(eval_iter):
    x_train, y_train = get_batch("test")
    pred = model(x_train)
    B, T, C = pred.shape
    pred = pred.view(B*T, C)
    y_train = y_train.view(B * T)
    loss = F.cross_entropy(pred, y_train)
    losses[i] = loss
  model.train()
  return losses.mean()

In [52]:
################
# BUILD MODELS #
################

In [53]:
class BigramModel(nn.Module):

  def __init__(self):
    super().__init__()
    self.embd = nn.Embedding(vocab_size, vocab_size)
  def forward(self, inputs, targets=None):
    x = self.embd(inputs)
    if targets is not None:
      B, T, C = x.shape
      targets = targets.view(B*T)
      loss = F.cross_entropy(x.view(B*T,C), targets)
      return x, loss
    return x, None

  def generate(self, start_token, n_tokens):
    for _ in range(n_tokens):
      input = start_token[-block_size:]                     # (B, T, C)
      logits, loss = self(input)                            # (B, T, C)
      logits = logits[:, -1, :]                             # (B, 1, C)
      probs = nn.functional.softmax(logits, dim=-1)         # (B, 1, C)
      idx = torch.multinomial(probs, num_samples=1)         # (B, 1)
      start_token = torch.cat((input, idx), dim=-1)         # (B, T+1)

    return start_token


In [54]:
emb = nn.Embedding(vocab_size, vocab_size)
preds = emb(torch.tensor(0).unsqueeze(0))
probs = nn.functional.softmax(preds, -1)
idx = torch.multinomial(probs, 1)
preds.shape, probs.shape

(torch.Size([1, 65]), torch.Size([1, 65]))

In [55]:
model = BigramModel()
start_token = torch.tensor(encode(" "))
token_ids = model.generate(start_token.unsqueeze(0), 100).squeeze().tolist()
decode(token_ids)

' F-fwPmK\nM??IUHgDUfaxs,:TL!fKjXY$asX&GR.g.-rD-TwGeVes!GkMBlsTqiQez\n.3KqiOaxUdKF$E,dNbZldD;hCssbc,?vLv'

In [56]:
class MultiheadAttention(nn.Module):

  def __init__(self):
    super().__init__()
    self.head_size = embed_dim // n_heads
    self.query = nn.Linear(embed_dim, self.head_size)
    self.key = nn.Linear(embed_dim, self.head_size)
    self.value = nn.Linear(embed_dim, self.head_size)
    self.buffer = None

  def forward(self, x):
    B, T, C = x.shape
    query = self.query
    key = self.key
    value = self.value
    x = query @ torch.transpose(key, 1, -1)
    x = (x / (self.head_size**-0.5))
    mask = torch.tril(torch.ones(8,8))
    x = x.masked_fill(mask[:8, :8] == 0 ,float("-inf"))
    x = x @ value
    return x

In [57]:
class TheGodamnTransformer(nn.Module):

  def __init__(self):
    super().__init__()
    self.embed = nn.Embedding(vocab_size, embed_dim)
    self.positional_encoding = nn.Embedding(block_size, embed_dim)

  def forward(self, x):
    x = self.embed(x) # B, T, C
    x = x + self.positional_encoding(torch.arange(0, block_size, device=device))



    return x

In [58]:
transformer = TheGodamnTransformer()
input = torch.tensor(1).unsqueeze(0)
output = transformer(input)
output.shape

torch.Size([8, 32])

In [59]:
mask = torch.tril(torch.ones(8,8))
mask = mask.masked_fill(mask[:8, :8] == 0 ,float("-inf"))
torch.nn.functional.softmax(mask, dim=-1)


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])